# Трек 2: SAGA — сокращение дисперсии без внешних циклов

SAGA хранит последнюю версию индивидуального градиента для каждого объекта и поддерживает среднее таблицы рекуррентно:

$$\bar G^{k+1}=\bar G^k + \frac{G_i^{new}-G_i^{old}}{m}.$$

Оценка градиента для loss-части:

$$g_k = \nabla f_i(x_k) - G_i^k + \bar G^k.$$

Для L2-регуляризации отдельно добавляется $\lambda x_k$, потому что это детерминированная часть полного градиента.

## Несмещенность

При равномерном выборе индекса $i$:

$$\mathbb E_i[g_k] = \frac1m\sum_{i=1}^m \nabla f_i(x_k) - \frac1m\sum_{i=1}^m G_i^k + \bar G^k = \nabla f(x_k).$$

Так как $\bar G^k = \frac1m\sum_i G_i^k$, две последние части сокращаются. Для регуляризованной задачи получаем $\mathbb E[g_k + \lambda x_k] = \nabla F(x_k)$.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
from src.experiments import add_residual, find_reference_minimum, make_team_oracles, plot_histories, plot_time_histories
from src.optimization import saga, svrg

FIGS = ROOT / 'figs'
FIGS.mkdir(exist_ok=True)

In [ ]:
# Для SAGA не берем слишком большой датасет: таблица градиентов имеет размер m x n.
oracle = make_team_oracles(m=1500, n=25, regcoef=3e-2, random_state=21)['Log-Cosh']
x0 = np.zeros(oracle.n)
x_star, f_star, opt_result = find_reference_minimum(oracle, x0)
print({'F*': f_star, 'table_memory_mb': oracle.m * oracle.n * 8 / 1024**2})

In [ ]:
histories = {}
_, _, h = svrg(
    oracle, x0, step_size=0.18, batch_size=32, max_epoch=70,
    inner_epochs=1, trace=True, random_state=5
)
histories['SVRG'] = add_residual(h, f_star)

_, _, h = saga(
    oracle, x0, step_size=0.08, batch_size=1, max_epoch=70,
    init_table=True, trace=True, random_state=5
)
histories['SAGA batch=1'] = add_residual(h, f_star)

_, _, h = saga(
    oracle, x0, step_size=0.10, batch_size=16, max_epoch=70,
    init_table=True, trace=True, random_state=5
)
histories['SAGA batch=16'] = add_residual(h, f_star)

plot_histories(histories, y_key='residual', logy=True, title='SAGA vs SVRG: эффективные эпохи')
plt.savefig(FIGS / '04_saga_vs_svrg_epochs.png', dpi=200, bbox_inches='tight')

plot_time_histories(histories, y_key='residual', logy=True, title='SAGA vs SVRG: время в секундах')
plt.savefig(FIGS / '04_saga_vs_svrg_time.png', dpi=200, bbox_inches='tight')

SVRG обычно дает ступенчатую траекторию из-за полного градиента в начале эпохи. SAGA тратит память `m x n`, зато обновляет оценку каждый шаг и часто дает более гладкую кривую. В сравнении по секундам заметны накладные расходы доступа к таблице.